In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, when, length

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []

    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = f'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/invoice_count'

invoice_file_paths = source_path + 'invoice_count/Invoice*.xlsx'

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(invoice_file_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Rename column name to avoid error at the time of write
combined_df = combined_df.rename(columns={"Payment block": "Payment_block","Doc.status":"Doc status",
                                          "Ref.key (header) 2":"Ref key (header) 2",
                                          "Ref.key (header) 1":"Ref key (header) 1"})

# Convert the Pandas DataFrame to a Spark DataFrame
invoice_df = spark.createDataFrame(combined_df)

# Cast all columns to StringType to avoid error at the time of write
invoice_df = invoice_df.select([col(c).cast("string") for c in invoice_df.columns])

# Add the new columns
invoice_df = invoice_df.withColumn("Payment_Categorization", 
    when(
        (length(col("Vendor")) < 7) & (col("Vendor").cast("int").isNotNull()), "PAYROLL/CONCUR"
    ).when(
        col("Vendor Account: Name 1").contains("EY PAYROLL"), "PAYROLL/CONCUR"
    ).when(
        col("Ref key (header) 2").rlike("ZEY_DOC|ZEY_PO_HLD"), "VIM"
    ).when(
        col("Vendor").isin(6400001, 6400002, 6400003), "CLAIMS"
    ).when(
        col("Logical System").contains("FuelPlus") | col("Text").contains("Fuel"), "FUEL"
    ).when(
        col("Logical System").contains("AMOS"), "VIM_AMOS"
    ).when(
        col("Logical System").contains("IATA"), "IATA / SIS"
    ).when(
        col("Logical System").contains("AirVision"), "AIRVISION"
    ).when(
        col("Text").rlike("Freight|Shipment"), "Freight"
    ).when(
        col("Text").rlike("Telephone|Electricity|Water|Internet|DEWA"), "Utility"
    ).when(
        col("Document Header Text").contains("Passenger Tax"), "Passenger Tax"
    ).when(
        col("Text").contains("Legal"), "Legal"
    ).when(
        col("Text").rlike("Commission|Incentives"), "Commission"
    ).when(
        col("Text").contains("Tax"), "Tax authority"
    ).when(
        col("Text").contains("Insurance"), "Insurance"
    ).when(
        col("Document type") == "SI", "Intercompany"
    ).when(
        col("Document type").isin("KC", "KG"), "Credit Note"
    ).when(
        col("Logical System").contains("REVERA"), "REVERA"
    ).otherwise('Unassigned')
)

# Write the data to the destination
invoice_df.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")